In [1]:
from ctypes import CDLL, c_uint32
from pathlib import Path

library_path = Path.cwd() / "slascone-client" / "SlasconeClient.so"
if not library_path.is_file():
    raise FileNotFoundError(f"Slascone client library not found: {library_path}")

slascone_client = CDLL(str(library_path))
slascone_client.slascone_client_abi_version.restype = c_uint32
slascone_client.slascone_client_abi_version.argtypes = []
abi_version = slascone_client.slascone_client_abi_version()
print(f"Loaded {library_path} (ABI version {abi_version})")

Loaded /workspace/slascone-client/SlasconeClient.so (ABI version 1)


### Using the Python client

`SlasconeClient` owns the native handle and closes it when the `with` block ends. The setters and the license check return a `SlasconeStatus`.

The ABI probe above only checks that the library loads. The bundled `.so` was observed to crash in `client_create` in the Jupyter Linux image, even from a C program. Use a working ABI 1 build via `library_path` before running this example.

```python
from pathlib import Path
from slascone_client import SlasconeClient, SlasconeStatus

with SlasconeClient("your-product-id", library_path="/path/to/SlasconeClient.so") as client:
    for configure, value in (
        (client.set_client_id, "your-client-id"),
        (client.set_software_release, "1.0"),
        (client.set_pem_key, Path("public-key.pem").read_text(encoding="utf-8")),
    ):
        status = configure(value)
        if status != SlasconeStatus.OK:
            raise RuntimeError(f"{configure.__name__}: {status.name}")

    print(client.check_offline_license("license.xml").name)  # Inline activation
```

To use a separate activation file, pass its path as the second argument to `check_offline_license`.

#### Extract the container ID from `/proc/self/mountinfo`

In [2]:
with open('/proc/self/mountinfo') as file:
    line = file.readline().strip()
    while line:
        if '/docker/containers/' in line:
            containerID = line.split('/docker/containers/')[-1]
            containerID = containerID.split('/')[0]
            print(f"Container ID: {containerID}")
        line = file.readline().strip()

Container ID: 090a258e5d0949f1cf12247ccea36a8bf82bc2c538d8b6c5757498a653a31284
Container ID: 090a258e5d0949f1cf12247ccea36a8bf82bc2c538d8b6c5757498a653a31284
Container ID: 090a258e5d0949f1cf12247ccea36a8bf82bc2c538d8b6c5757498a653a31284


In [1]:
from pathlib import Path
from slascone_client import SlasconeClient, SlasconeStatus

library_path = Path.cwd() / "slascone-client" / "SlasconeClient.so"

public_key = """-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAwpigzm+cZIyw6x253YRD
mroGQyo0rO9qpOdbNAkE/FMSX+At5CQT/Cyr0eZTo2h+MO5gn5a6dwg2SYB/K1Yt
yuiKqnaEUfoPnG51KLrj8hi9LoZyIenfsQnxPz+r8XGCUPeS9MhBEVvT4ba0x9Ew
R+krU87VqfI3KNpFQVdLPaZxN4STTEZaet7nReeNtnnZFYaUt5XeNPB0b0rGfrps
y7drmZz81dlWoRcLrBRpkf6XrOTX4yFxe/3HJ8mpukuvdweUBFoQ0xOHmG9pNQ31
AHGtgLYGjbKcW4xYmpDGl0txfcipAr1zMj7X3oCO9lHcFRnXdzx+TTeJYxQX2XVb
hQIDAQAB
-----END PUBLIC KEY-----"""

license_file = Path.cwd() / "assets" / "License_1eb98e63-1146-4497-a4a7-7e4617506d27.xml"
activation_file = Path.cwd() / "assets" / "ActivationFile.xml"

with SlasconeClient("b18657cc-1f7c-43fa-e3a4-08da6fa41ad3", library_path=library_path) as client:
    for configure, value in (
        (client.set_client_id, "cd9f7f73da44"),
        (client.set_software_release, "26.3"),
        (client.set_pem_key, public_key),
    ):
        status = configure(value)
        if status != SlasconeStatus.OK:
            raise RuntimeError(f"{configure.__name__}: {status.name}")

    print(client.check_offline_license(license_file, activation_file).name)  # Inline activation

OK
